In [ ]:
"""Get regions of 100kb signals with most variable signals across biospecimens"""
# pylint: disable=redefined-outer-name, import-error

In [ ]:
%load_ext autoreload
%autoreload 2

## SETUP

In [ ]:
import json
import os
import re
import subprocess
import tarfile
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import pandas as pd
from gprofiler import GProfiler  # gprofiler-official package
from IPython.display import display

from epiclass.utils.bed_utils import bins_to_bed

In [ ]:
CELL_TYPE: str = "harmonized_sample_ontology_intermediate"
ASSAY: str = "assay_epiclass"

In [ ]:
paper_dir = Path.home() / "Projects/epiclass/output/paper"
if not paper_dir.exists():
    raise FileNotFoundError(f"Paper directory not found: {paper_dir}")

In [ ]:
data_dir = paper_dir / "data"
metadata_dir = data_dir / "metadata"
if not metadata_dir.exists():
    raise FileNotFoundError(f"Metadata directory not found: {metadata_dir}")

In [ ]:
input_dir = (
    paper_dir / "data" / "differential_regions" / "epiatlas_dfreeze_v2.1_16ct_pval"
)
if not input_dir.exists():
    raise FileNotFoundError()

In [ ]:
output_dir = input_dir / "analysis_files"
if not output_dir.exists():
    output_dir.mkdir()

## Compute most variable regions

### Metadata

In [ ]:
def normalize_string(s: str) -> str:
    """Normalize a string by stripping whitespace, converting to lowercase,
    and replacing spaces and hyphens by underscore.
    """
    s = s.strip().lower()
    s = re.sub(r"[\s-]+", "_", s)
    return s

In [ ]:
meta_path = (
    metadata_dir / "epiatlas" / "hg38_2023-epiatlas-dfreeze_v2.1_w_encode_noncore_2.json"
)
with open(meta_path, "r", encoding="utf8") as f:
    meta = json.load(f)

meta_df = pd.DataFrame.from_records(meta["datasets"])
print(meta_df.shape)
display(meta_df.value_counts(CELL_TYPE))

In [ ]:
# remove ENCODE samples
meta_df = meta_df[~meta_df[CELL_TYPE].isna()]
print(meta_df.shape)

In [ ]:
meta_df[CELL_TYPE] = meta_df[CELL_TYPE].apply(normalize_string)
meta_df.value_counts(CELL_TYPE)

### Load 100 kb signals

In [ ]:
data = np.load(input_dir / "epiatlas_dfreeze_v2.1_16ct_pval.npz", allow_pickle=True)

In [ ]:
arr = data["signals"]
ids = data["ids"]

print(arr.shape)
print(ids.shape)

Filter metadata

In [ ]:
meta_df = meta_df[meta_df["md5sum"].isin(ids)]
assert len(meta_df) == len(ids)

In [ ]:
meta_df = meta_df.set_index("md5sum").loc[ids]
biospecimens = meta_df.loc[:, CELL_TYPE].to_list()

In [ ]:
display(meta_df[CELL_TYPE].value_counts())
display(meta_df[ASSAY].value_counts())

cell_types = list(meta_df[CELL_TYPE].unique())
chip_assays = list(meta_df[ASSAY].unique())

In [ ]:
assert list(meta_df.index) == list(ids), "Index mismatch after reordering"

In [ ]:
# Create labels for both biospecimen and assay
biospecimen_labels = np.array(meta_df[CELL_TYPE].to_list())
assay_labels = np.array(meta_df[ASSAY].to_list())

unique_biospecimens = np.unique(biospecimen_labels)
unique_assays = np.unique(assay_labels)

print(f"Signal shape: {arr.shape}")
print(f"Unique biospecimens: {len(unique_biospecimens)}")
print(f"Unique assays: {len(unique_assays)}")

### Compute effect sizes

#### Per biospecimen

In [ ]:
# Compare mean of this biospecimen vs mean/std of all other biospecimens
effect_sizes_bio = np.zeros((len(unique_biospecimens), arr.shape[1]))

for i, bio in enumerate(unique_biospecimens):
    mask = biospecimen_labels == bio

    this_mean = arr[mask].mean(axis=0)
    rest_mean = arr[~mask].mean(axis=0)
    rest_std = np.maximum(arr[~mask].std(axis=0), 1e-8)

    effect_sizes_bio[i] = (this_mean - rest_mean) / rest_std

In [ ]:
# Wrap in a DataFrame for easier inspection
effect_df_bio = pd.DataFrame(
    effect_sizes_bio,
    index=unique_biospecimens,
    columns=[f"bin_{i}" for i in range(effect_sizes_bio.shape[1])],
)
print(f"Number of null regions: {effect_df_bio.isnull().sum().sum()}")  # Check for NaNs
print(effect_df_bio.shape)
display(effect_df_bio.head())

#### Per (assay, biospecimen) pair

In [ ]:
def compute_effect_sizes_for_assay(arr, biospecimen_labels, assay_labels, target_assay):
    """
    For a given assay, compute effect sizes for each biospecimen
    using one-vs-rest within that assay.
    """
    # Filter to samples of this assay
    assay_mask = assay_labels == target_assay
    arr_assay = arr[assay_mask]
    bio_labels_assay = biospecimen_labels[assay_mask]

    unique_bios = np.unique(bio_labels_assay)

    effect_sizes = {}

    for bio in unique_bios:
        mask = bio_labels_assay == bio

        # Skip if too few samples
        if mask.sum() < 5 or (~mask).sum() < 5:
            print(
                f"Skipping {target_assay}/{bio}: insufficient samples ({mask.sum()} vs {(~mask).sum()})"
            )
            continue

        this_mean = arr_assay[mask].mean(axis=0)
        rest_mean = arr_assay[~mask].mean(axis=0)
        rest_std = np.maximum(arr_assay[~mask].std(axis=0), 1e-8)

        effect_sizes[bio] = (this_mean - rest_mean) / rest_std

    return effect_sizes

In [ ]:
# Compute effect sizes for all (assay, biospecimen) pairs
pairs_effect_sizes = {}  # key: (assay, biospecimen), value: effect size array

for assay in unique_assays:
    print(f"Processing assay: {assay}")
    assay_effects = compute_effect_sizes_for_assay(
        arr, biospecimen_labels, assay_labels, assay
    )
    for bio, effects in assay_effects.items():
        pairs_effect_sizes[(assay, bio)] = effects

print(f"\nTotal (assay, biospecimen) pairs: {len(pairs_effect_sizes)}")

In [ ]:
# # Convert to DataFrame for easier inspection
# effect_df_pairs = pd.DataFrame(
#     {f"{assay}_{bio}": effects for (assay, bio), effects in pairs_effect_sizes.items()}
# ).T
# effect_df_pairs.columns = [f"bin_{i}" for i in range(effect_df_pairs.shape[1])]

# print(f"Effect sizes shape: {effect_df_pairs.shape}")
# print(f"Number of null regions: {effect_df_pairs.isnull().sum().sum()}")
# display(effect_df_pairs.head())

#### Per (assay, biospecimen) pair - Global background

In [ ]:
def compute_effect_sizes_global_background(
    arr: np.ndarray,
    biospecimen_labels: np.ndarray,
    assay_labels: np.ndarray,
) -> Dict[Tuple[str, str], np.ndarray]:
    """
    For each (assay, biospecimen) pair, compute effect size where:
    - this = samples of this assay AND this biospecimen
    - rest = ALL samples that are NOT this biospecimen (any assay)

    This more closely mirrors SHAP's global background approach.
    """
    unique_assays = np.unique(assay_labels)
    unique_bios = np.unique(biospecimen_labels)

    effect_sizes = {}

    for assay in unique_assays:
        print(f"Processing {assay}")
        for bio in unique_bios:
            # this = specific (assay, biospecimen) pair
            mask_this = (assay_labels == assay) & (biospecimen_labels == bio)

            # rest = all samples NOT of this biospecimen (across all assays)
            mask_rest = biospecimen_labels != bio

            n_this = mask_this.sum()
            if n_this < 5:
                print(f"Skipping {assay}/{bio}: insufficient samples ({n_this})")
                continue

            this_mean = arr[mask_this].mean(axis=0)
            rest_mean = arr[mask_rest].mean(axis=0)
            rest_std = np.maximum(arr[mask_rest].std(axis=0), 1e-8)

            effect_sizes[(assay, bio)] = (this_mean - rest_mean) / rest_std

    return effect_sizes

In [ ]:
pairs_effect_sizes_global = compute_effect_sizes_global_background(
    arr, biospecimen_labels, assay_labels
)

print(
    f"Total (assay, biospecimen) pairs with global background: {len(pairs_effect_sizes_global)}"
)

### Get SHAP regions count

In [ ]:
shap_dir = (
    data_dir
    / "SHAP"
    / "hg38_100kb_all_none/harmonized_sample_ontology_intermediate_1l_3000n/10fold-oversampling"
)
if not shap_dir.exists():
    raise FileNotFoundError(f"SHAP directory {shap_dir} does not exist.")

biospecimen_shap_count = {}
assay_biospecimen_shap_count = {}

shap_gprofiler_results = {}

shap_path = shap_dir / "select_beds_top303.tar.gz"
with tarfile.open(shap_path, "r:gz") as tar:
    for member in tar.getmembers():
        if not member.name.endswith("bed"):
            continue

        filename = member.name.split("/")[1]
        group_name = filename.replace("_features.bed", "")
        group_name = normalize_string(group_name)

        # per biospecimen overall
        if "merge_samplings_" in group_name:
            # print(f"Extracting {member.name} from tarball")

            # Extract biospecimen name
            biospecimen = group_name.replace("merge_samplings_", "")
            print(f"Processing: {biospecimen}")

            # # Read the file into a DataFrame
            f = tar.extractfile(member)
            if f is not None:
                df = pd.read_csv(f, sep="\t", header=None)
                count = df.shape[0]
                biospecimen_shap_count[biospecimen] = count
                # print(f"Number of SHAP regions for {biospecimen}: {count}")

            gprofiler_name = member.name.replace(".bed", "_intersect_gff_gprofiler.tsv")
            f = tar.extractfile(gprofiler_name)
            df = pd.read_csv(f, sep="\t", header=0, low_memory=False)  # type: ignore
            shap_gprofiler_results[biospecimen] = df
            continue

        # Per (assay, biospecimen) pair
        if any(group_name.startswith(assay) for assay in unique_assays):
            assay, biospecimen = group_name.split("_", maxsplit=1)
            print(f"Processing: ({assay}, {biospecimen})")

            f = tar.extractfile(member)
            if f is not None:
                df = pd.read_csv(f, sep="\t", header=None)
                count = df.shape[0]
                assay_biospecimen_shap_count[(assay, biospecimen)] = count

            gprofiler_name = member.name.replace(".bed", "_intersect_gff_gprofiler.tsv")
            f = tar.extractfile(gprofiler_name)
            df = pd.read_csv(f, sep="\t", header=0, low_memory=False)  # type: ignore
            shap_gprofiler_results[f"{assay}_{biospecimen}"] = df

            continue

### Select regions

#### Per biospecimen

In [ ]:
def get_top_bins_variable(effect_df, n_per_bio):
    """
    n_per_bio: dict of {biospecimen: n_regions}
    Returns dict of {biospecimen: list of top bin indices}
    """
    top_bins = {}
    for bio, n in n_per_bio.items():
        if bio not in effect_df.index:
            print(f"Warning: {bio} not in effect_df")
            continue
        abs_effects = effect_df.loc[bio].abs()
        top_bins[bio] = abs_effects.nlargest(n).index.tolist()
    return top_bins

In [ ]:
top_bins_bio = get_top_bins_variable(effect_df_bio, biospecimen_shap_count)

for biospecimen, bins in top_bins_bio.items():
    print(f"{biospecimen}: {len(bins)} regions")
    assert len(bins) == biospecimen_shap_count[biospecimen]

#### Per (assay, biospecimen) pairs

In [ ]:
def get_top_bins_variable_pairs(
    effect_sizes: Dict[Tuple[str, str], np.ndarray],
    n_per_pair: Dict[Tuple[str, str], int],
):
    """
    effect_sizes: dict of {(assay, biospecimen): effect_array}
    n_per_pair: dict of {(assay, biospecimen): n_regions}
    Returns dict of {(assay, biospecimen): list of top bin indices}
    """
    top_bins = {}

    for (assay, bio), n in n_per_pair.items():
        key = (assay, bio)

        if key not in effect_sizes:
            print(
                f"Warning: {assay}/{bio} not in effect_sizes (no samples or filtered out)"
            )
            continue

        effects = effect_sizes[key]
        abs_effects = np.abs(effects)
        top_indices = np.argsort(abs_effects)[::-1][:n]
        top_bins[key] = [f"bin_{i}" for i in top_indices]

    return top_bins

In [ ]:
top_bins_pairs_global = get_top_bins_variable_pairs(
    pairs_effect_sizes_global, assay_biospecimen_shap_count
)

top_bins_pairs_1 = get_top_bins_variable_pairs(
    pairs_effect_sizes, assay_biospecimen_shap_count
)

verbose = False
# Verify counts match
for pairs_dict in [top_bins_pairs_global, top_bins_pairs_1]:
    for (assay, bio), bins in pairs_dict.items():
        expected = assay_biospecimen_shap_count[(assay, bio)]
        actual = len(bins)
        if actual != expected:
            print(f"Mismatch for {assay}/{bio}: expected {expected}, got {actual}")
        else:
            if verbose:
                print(f"{assay}/{bio}: {actual} regions ✓")

### Write to files

In [ ]:
bed_folder_bio = output_dir / "per_biospecimen"
bed_folder_bio.mkdir(exist_ok=True)

bed_folder_pairs = output_dir / "per_assay_biospecimen_pair"
bed_folder_pairs.mkdir(exist_ok=True)

bed_folder_pairs_global = output_dir / "per_assay_biospecimen_pair_global"
bed_folder_pairs_global.mkdir(exist_ok=True)

In [ ]:
chrom_file = data_dir / "chromsizes" / "hg38.noy.chrom.sizes"
if not chrom_file.exists():
    raise FileNotFoundError(f"Chromosome sizes file not found: {chrom_file}")

with open(chrom_file, "r", encoding="utf8") as f:
    chrom_sizes = []
    for line in f:
        chrom, size = line.strip().split("\t")
        chrom_sizes.append((chrom, int(size)))

Per biospecimen

In [ ]:
first_key = list(top_bins_bio.keys())[0]

if isinstance(top_bins_bio[first_key][0], str):
    for bio, bins in list(top_bins_bio.items()):
        bins_as_int = [int(b.replace("bin_", "")) for b in bins]
        top_bins_bio[bio] = sorted(bins_as_int)

In [ ]:
with open(
    output_dir / "top_variable_regions_per_biospecimen_N_SHAP.json", "w", encoding="utf8"
) as f:
    json.dump(top_bins_bio, f, indent=2)

In [ ]:
for bio, bins in top_bins_bio.items():
    output_name = bed_folder_bio / f"{bio}_top_variable_regions_N_SHAP.bed"
    bins_to_bed(
        bin_indexes=bins,
        chroms=chrom_sizes,
        resolution=100 * 1000,
        sort=True,
        bed_path=output_name,
        verbose=False,
    )
    # print(f"Wrote '{output_name}'")

Per (assay, biospecimen) pair

In [ ]:
pairs_folder = bed_folder_pairs_global
top_pairs = top_bins_pairs_global

In [ ]:
first_key = list(top_pairs.keys())[0]

if isinstance(top_pairs[first_key][0], str):
    for key, bins in list(top_pairs.items()):
        bins_as_int = [int(b.replace("bin_", "")) for b in bins]
        top_pairs[key] = sorted(bins_as_int)

In [ ]:
# Save as JSON (convert tuple keys to strings)
json_output_pairs = {f"{assay}__{bio}": bins for (assay, bio), bins in top_pairs.items()}

json_name = "top_variable_regions_N_SHAP_by_assay_biospecimen_pair"
if top_pairs == top_bins_pairs_1:
    json_name = json_name + ".json"
elif top_pairs == top_bins_pairs_global:
    json_name = json_name + "_global_bg.json"
else:
    raise UserWarning("Unexpected input bins object.")

with open(output_dir / json_name, "w", encoding="utf8") as f:
    json.dump(json_output_pairs, f, indent=2)

In [ ]:
for (assay, bio), bins in top_pairs.items():
    output_name = pairs_folder / f"{assay}_{bio}_top_variable_regions_N_SHAP.bed"
    bins_to_bed(
        bin_indexes=bins,
        chroms=chrom_sizes,
        resolution=100 * 1000,
        sort=True,
        bed_path=output_name,
        verbose=False,
    )

## gprofiler results

In [ ]:
bedtools_path = "/usr/local/bin/bedtools"  # Adjust as needed
gff_path = data_dir / "regions" / "gff" / "Homo_sapiens.GRCh38.109.chr.filtered.gff3"
if not gff_path.exists():
    raise FileNotFoundError()

### Find genes intersection

In [ ]:
def intersect_one_bed(input_bed_path: Path, output_filename: Path, verbose=False) -> None:
    """Intersect bed with gene-gff. Does nothing if output_filename already exists."""
    # don't redo work
    if output_filename.is_file():
        if verbose:
            print(f"{output_filename} already exists.")
        return

    cmd = [
        str(bedtools_path),
        "intersect",
        "-a",
        str(input_bed_path),
        "-b",
        str(gff_path),
        "-wo",
        "-F",
        "0.5",
    ]
    output = subprocess.check_output(cmd).decode()

    if verbose:
        print(f"Writing to {output_filename}")

    with open(output_filename, "w", encoding="utf8") as out:
        out.writelines(output)

In [ ]:
for folder in [bed_folder_bio, bed_folder_pairs]:
    for bed_file in folder.glob("*.bed"):
        # Empty file, delete
        if os.stat(str(bed_file)).st_size == 0:
            os.remove(str(bed_file))
            continue

        # Intersect
        output_name = Path(bed_file.stem + "_intersect_gff.tsv")
        output_filename = folder / output_name
        intersect_one_bed(bed_file, output_filename)

### Compute gprofiler results

In [ ]:
def run_gprofiler_on_folder(bed_folder: Path, verbose=False) -> None:
    """Run bedtools intersect and gprofiler on all bed files in folder."""
    gp = GProfiler(return_dataframe=True)

    # bedtools intersect
    for bed_file in bed_folder.glob("*.bed"):
        # Empty file, remove
        if os.stat(str(bed_file)).st_size == 0:
            os.remove(str(bed_file))
            continue

        # Run bedtools intersect
        output_name = Path(bed_file.stem + "_intersect_gff.tsv")
        output_filename = bed_folder / output_name
        intersect_one_bed(bed_file, output_filename, verbose)

    # gprofiler
    for intersect_file in bed_folder.glob("*_intersect_gff.tsv"):
        new_file = bed_folder / f"{intersect_file.stem}_gprofiler.tsv"
        # don't redo work
        if new_file.is_file():
            if verbose:
                print(f"{new_file} already exists.")
            continue

        try:
            intersect_df = pd.read_csv(intersect_file, sep="\t", header=None)
        except pd.errors.EmptyDataError:
            print(f"Empty intersect: {intersect_file}\n")
            continue

        genes = intersect_df[11].str.extract(r"ID=gene:(\w+);").drop_duplicates()
        genes_list = sorted(genes[0].values)

        gene_list_path = bed_folder / f"{intersect_file.stem}_genes.list"
        with open(gene_list_path, "w", encoding="utf8") as out:
            out.write("\n".join(genes_list))

        if verbose:
            print(f"Writing GO results to {new_file}")
        go_profile: pd.DataFrame = gp.profile(query=genes_list)  # type: ignore
        go_profile.to_csv(new_file, sep="\t", index=False)

In [ ]:
for folder in [bed_folder_bio, bed_folder_pairs, bed_folder_pairs_global]:
    print(f"Processing {folder.name}")
    run_gprofiler_on_folder(folder)

### Format gprofiler results

In [ ]:
def parse_biospecimen_from_filename(name: str) -> Dict[str, str]:
    """Parse biospecimen from per-biospecimen filename."""
    match = re.match(r"(.*)_top_variable.*", name)
    if match:
        return {"biospecimen": match.group(1)}
    print(f"Warning: Could not parse {name}")
    return {"biospecimen": "unknown"}


def parse_assay_biospecimen_from_filename(name: str) -> Dict[str, str]:
    """Parse assay and biospecimen from per-(assay, biospecimen) filename."""
    match = re.match(r"([^_]+)_(.+?)_top_variable_regions.*", name)
    if match:
        return {"assay": match.group(1), "biospecimen": match.group(2)}
    print(f"Warning: Could not parse {name}")
    return {"assay": "unknown", "biospecimen": "unknown"}

In [ ]:
def format_gprofiler_results(
    bed_folder: Path, parser_func, output_path: Path, verbose: bool = False
) -> pd.DataFrame:
    """
    Load, format, and concatenate gprofiler results from a folder.

    Args:
        bed_folder: Path to folder containing *_gprofiler.tsv files
        parser_func: Function that takes filename and returns dict of metadata columns
        output_path: Path to save concatenated results

    Returns:
        Concatenated DataFrame with all GO results
    """
    all_go_dfs = {}

    for go_file in bed_folder.glob("*_gprofiler.tsv"):
        go_df = pd.read_csv(go_file, sep="\t")
        all_go_dfs[go_file.stem] = go_df
        if verbose:
            print(f"{go_file.stem}: {go_df.shape}")

    for name, df in list(all_go_dfs.items()):
        if df.empty:
            print(f"{name}: gprofiler results empty, deleting from dict")
            del all_go_dfs[name]
            continue

        # Add metadata columns from filename
        metadata = parser_func(name)
        for col, val in metadata.items():
            df.loc[:, col] = val

        df.loc[:, "-log10(p_value)"] = -np.log10(df.loc[:, "p_value"])
        all_go_dfs[name] = df

    if not all_go_dfs:
        print("Warning: No non-empty gprofiler results found")
        return pd.DataFrame()

    full_df = pd.concat(all_go_dfs.values())
    full_df = full_df.drop(["significant", "query"], axis=1)

    full_df.to_csv(output_path, sep="\t", index=False)
    if verbose:
        print(f"Saved to {output_path}")
        print(f"Shape: {full_df.shape}")
        display(full_df.head())

    return full_df

Per biospecimen

In [ ]:
concat_per_bio = format_gprofiler_results(
    bed_folder=bed_folder_bio,
    parser_func=parse_biospecimen_from_filename,
    output_path=output_dir / "GO_biospecimen_table.tsv",
)

Per (assay, biospecimen) pairs

In [ ]:
concat_pairs = format_gprofiler_results(
    bed_folder=bed_folder_pairs,
    parser_func=parse_assay_biospecimen_from_filename,
    output_path=output_dir / "GO_assay_biospecimen_table.tsv",
)

Per (assay, biospecimen) pairs - global background

In [ ]:
concat_pairs_global = format_gprofiler_results(
    bed_folder=bed_folder_pairs_global,
    parser_func=parse_assay_biospecimen_from_filename,
    output_path=output_dir / "GO_assay_biospecimen_global_bg_table.tsv",
)

SHAP regions

In [ ]:
per_bio_shap_dfs = []

for bio in unique_biospecimens:
    try:
        df = shap_gprofiler_results[bio]
        df["biospecimen"] = bio
        df["-log10(p_value)"] = -np.log10(df["p_value"])
        per_bio_shap_dfs.append(df)
    except KeyError as err:
        print(f"Skipping missing: {bio} ({err})")

concat_per_bio_shap_df = pd.concat(per_bio_shap_dfs)

In [ ]:
per_pair_shap_dfs = []

for assay, bio in assay_biospecimen_shap_count:
    try:
        key = f"{assay}_{bio}"
        df = shap_gprofiler_results[key]
        df["assay"] = assay
        df["biospecimen"] = bio
        df["-log10(p_value)"] = -np.log10(df["p_value"])
        per_pair_shap_dfs.append(df)
    except KeyError as err:
        print(f"Skipping missing: ({assay}, {bio}) ({err})")

concat_per_pair_shap_df = pd.concat(per_pair_shap_dfs)

### Quantify gprofiler results

In [ ]:
def compute_metrics(df: pd.DataFrame):
    """Compute avg/std, median/IQR, for grofiler pvalues."""
    N = df.shape[0]
    label = "-log10(p_value)"
    if label not in df:
        df[label] = df["table_val"]

    col = df[label]
    mean = col.mean()
    std = col.std()
    med = col.median()
    iqr = col.quantile(0.75) - col.quantile(0.25)
    print(
        f"N: {N}\nMean: {mean:.2f}\nStd: {std:.2f}\nMedian: {med:.2f}\nIQR: {iqr:.2f}\n"
    )

In [ ]:
selected_cell_types = [
    "t_cell",
    "neutrophil",
    "lymphocyte_of_b_lineage",
    "brain",
    "hepatocyte",
]

In [ ]:
print("Reference pvalues/-log10:")
for val in [0.05, 0.01, 0.001, 0.0001]:
    print(
        f"{val}: {-np.log10(val):.2f}",
    )
print("---")

for name, df in zip(
    [
        "SHAP per bio (merge_samplings)",
        "Effect size per bio",
        "SHAP per pair (assay_bio)",
        "Effect size per pair, global bg",
    ],
    [
        concat_per_bio_shap_df,
        concat_per_bio,
        concat_per_pair_shap_df,
        concat_pairs_global,
    ],
):
    print(f"Global results {name}")
    compute_metrics(df)

    print("Filtered to selected cell types")
    sub_df = df[df["biospecimen"].isin(selected_cell_types)]
    compute_metrics(sub_df)
    print("-----")

In [ ]:
def compute_metrics_extended(df: pd.DataFrame, name: str, scope: str) -> list[Dict]:
    """Compute extended metrics for gprofiler results at multiple aggregation levels."""
    label = "-log10(p_value)"
    if label not in df.columns:
        df = df.copy()
        df[label] = -np.log10(df["p_value"])

    col = df[label]

    base_info = {
        "Method": name,
        "Scope": scope,
    }

    results = []

    # View 1: All (biospecimen, term) pairs
    results.append(
        {
            **base_info,
            "Aggregation": "All (bio, term) pairs",
            "N": len(df),
            "N unique terms": df["native"].nunique()
            if "native" in df.columns
            else np.nan,
            "N biospecimens": df["biospecimen"].nunique(),
            "Mean terms per bio": df.groupby("biospecimen").size().mean(),
            "% p < 0.05": (df["p_value"] < 0.05).mean() * 100,
            "% p < 0.01": (df["p_value"] < 0.01).mean() * 100,
            "% p < 0.001": (df["p_value"] < 0.001).mean() * 100,
            "Mean -log10(p)": col.mean(),
            "Median -log10(p)": col.median(),
            "Std -log10(p)": col.std(),
            "IQR -log10(p)": col.quantile(0.75) - col.quantile(0.25),
            "Max -log10(p)": col.max(),
            "90th percentile": col.quantile(0.90),
        }
    )

    # View 2: Per unique term (best p-value across biospecimens)
    best_per_term = df.groupby("native")[label].max()
    pval_per_term = df.groupby("native")["p_value"].min()
    results.append(
        {
            **base_info,
            "Aggregation": "Per unique term (best p)",
            "N": len(best_per_term),
            "N unique terms": len(best_per_term),
            "N biospecimens": np.nan,
            "Mean terms per bio": np.nan,
            "% p < 0.05": (pval_per_term < 0.05).mean() * 100,
            "% p < 0.01": (pval_per_term < 0.01).mean() * 100,
            "% p < 0.001": (pval_per_term < 0.001).mean() * 100,
            "Mean -log10(p)": best_per_term.mean(),
            "Median -log10(p)": best_per_term.median(),
            "Std -log10(p)": best_per_term.std(),
            "IQR -log10(p)": best_per_term.quantile(0.75) - best_per_term.quantile(0.25),
            "Max -log10(p)": best_per_term.max(),
            "90th percentile": best_per_term.quantile(0.90),
        }
    )

    # View 3: Per biospecimen (median across terms)
    median_per_bio = df.groupby("biospecimen")[label].median()
    pval_median_per_bio = df.groupby("biospecimen")["p_value"].median()
    results.append(
        {
            **base_info,
            "Aggregation": "Per biospecimen (median)",
            "N": len(median_per_bio),
            "N unique terms": np.nan,
            "N biospecimens": len(median_per_bio),
            "Mean terms per bio": np.nan,
            "% p < 0.05": (pval_median_per_bio < 0.05).mean() * 100,
            "% p < 0.01": (pval_median_per_bio < 0.01).mean() * 100,
            "% p < 0.001": (pval_median_per_bio < 0.001).mean() * 100,
            "Mean -log10(p)": median_per_bio.mean(),
            "Median -log10(p)": median_per_bio.median(),
            "Std -log10(p)": median_per_bio.std(),
            "IQR -log10(p)": median_per_bio.quantile(0.75)
            - median_per_bio.quantile(0.25),
            "Max -log10(p)": median_per_bio.max(),
            "90th percentile": median_per_bio.quantile(0.90),
        }
    )

    return results

In [ ]:
results_extended = []

for name, df in [
    ("SHAP per bio (merge_samplings)", concat_per_bio_shap_df),
    ("Effect size per bio", concat_per_bio),
    ("SHAP per pair (assay_bio)", concat_per_pair_shap_df),
    ("Effect size per pair (global bg)", concat_pairs_global),
]:
    # All biospecimens
    results_extended.extend(compute_metrics_extended(df, name, "All biospecimens"))

    # Selected cell types only
    sub_df = df[df["biospecimen"].isin(selected_cell_types)]
    if not sub_df.empty:
        results_extended.extend(
            compute_metrics_extended(sub_df, name, "Selected cell types")
        )

results_extended_df = pd.DataFrame(results_extended)

In [ ]:
print("GO Enrichment Results Summary (Extended)")
print(f"Selected cell types: {', '.join(selected_cell_types)}")
print()
display(results_extended_df.round(2))

results_extended_df.to_csv(
    output_dir / "GO_enrichment_comparison_extended.tsv", sep="\t", index=False
)